In [37]:

import snowflake.connector as sc
import os, base64, hashlib, time, jwt, requests
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives import serialization
import json
from dotenv import load_dotenv
from snowflake.cortex import Complete
from snowflake.core import Root
import json
import re
from snowflake.snowpark import Session

load_dotenv()

snowflake_account = os.getenv("SNOWFLAKE_ACCOUNT")
snowflake_user = os.getenv("SNOWFLAKE_PROJECT_USER")
private_key = os.getenv("PRIVATE_KEY")



In [2]:
def encode_private_key(key:str):
    p_key= serialization.load_pem_private_key(
    key.encode("utf-8"),
    password=None
    )   
    pkb = p_key.private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption())
    return p_key, pkb

In [38]:
def snowflake_session(pkb,snowflake_account,snowflake_user):
    conn_params = { 
        'account': snowflake_account,
        'user': snowflake_user,
        'authenticator': 'SNOWFLAKE_JWT',
        'private_key': pkb
    }
    session = Session.builder.configs(conn_params).create() 
    return session
    
    

In [4]:
def execute_sql(sql:str,session:Session,max_rows:int=1000):
    try:
        df = session.sql(sql.replace(';','')).limit(max_rows).to_pandas()  
        return df
    except Exception as e:
        return f"Error executing SQL: {str(e)}"

In [5]:
def generate_jwt_token(p_key,snowflake_acount:str,snowflake_user:str):
    pub_der = p_key.public_key().public_bytes(serialization.Encoding.DER, serialization.PublicFormat.SubjectPublicKeyInfo)
    fp = "SHA256:" + base64.b64encode(hashlib.sha256(pub_der).digest()).decode()
    now = int(time.time())
    payload = {
        "iss": f"{snowflake_acount}.{snowflake_user}.{fp}",
        "sub": f"{snowflake_acount}.{snowflake_user}",
        "iat": now,
        "exp": now + 59 * 60,   # <= 1 hour
    }
    return jwt.encode(payload, p_key, algorithm="RS256")

In [6]:
def snowflake_agent_call(jwt_token:str,snowflake_account:str,prompt:str):
    payload = {
    "model": "claude-3-5-sonnet",
    "response_instruction": prompt,
    "messages": [          
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "For Gamersupps, what is AOV for the last 30 days?"
                }
            ]
        }
    ],
    "tools": [
        {
            "tool_spec": {
                "type": "cortex_analyst_text_to_sql",
                "name": "analyze_data"
            }
        }
        
    ],
    "tool_resources": {
        "analyze_data": {"semantic_model_file": "@OB_AI.CHATBOT.APP/client_analytics.yaml"}
    }
    } 
    url = f"https://{snowflake_account}.snowflakecomputing.com/api/v2/cortex/agent:run"
    headers = {
        "Authorization": f"Bearer {jwt_token}",
        "X-Snowflake-Authorization-Token-Type": "KEYPAIR_JWT",
        "Content-Type": "application/json"
    }
    try:
        resp = requests.post(url, headers=headers, json=payload)
        if resp.status_code == 200:
            with resp as r:
                for line in r.iter_lines():
                    if line:
                        decoded = line.decode("utf-8")
                        if decoded.startswith("data: "):                   
                            payload = decoded[6:]  # strip "data: "
                            try:
                                data = json.loads(payload)
                                yield data # Now you can parse this JSON chunk
                            except:
                                pass
        else:
            return 'Error: ' + resp.text
    except Exception as e:
        return 'Error: ' + str(e)

In [39]:
from snowflake.cortex import complete,classify_text

In [7]:
def execute_cortex_complete_api(prompt,session):    
    """
    Execute Cortex Complete using the REST API
    """
    response_txt = Complete(
                    'claude-3-5-sonnet',
                    prompt,
                    session=session
                    )
    return response_txt

In [8]:
p_key, pkb = encode_private_key(private_key)
jwt_token = generate_jwt_token(p_key,snowflake_account,snowflake_user)


In [40]:
session = snowflake_session(pkb,snowflake_account,snowflake_user)

In [13]:
event='text'
data= 'Interating with database'











In [46]:
json.loads(classify_text('what is today\'s temperature in tokyo?' ,[{
    'label': 'data',
    'description': 'questions that needs sql queries to solve. data available includes delivery time, financial numbers, order economics and marketing metrics, and retention metrics',
    'examples': ['what is my aov last month','which region is most profitable','what is my average shipping time']
  },{
    'label': 'general',
    'description': 'general questions that are not data related',
    'examples': ['why did my orders stuck in customs?', 'what does aov mean?','how do you define transaction amount?']
    }],session))

{'label': 'general'}

In [49]:
session.sql("SET brand = 'drmtlgy'").collect()

[Row(status='Statement executed successfully.')]

In [52]:
session.variables

AttributeError: 'Session' object has no attribute 'variables'

In [10]:
r1=snowflake_agent_call(jwt_token,snowflake_account,'you should always be friendly')


In [12]:
list(r1)

[{'id': 'msg_001',
  'object': 'message.delta',
  'delta': {'content': [{'index': 0,
     'type': 'tool_use',
     'tool_use': {'tool_use_id': 'tooluse_6a8ba8f98bd047d2bdaeed',
      'name': 'analyze_data',
      'type': 'cortex_analyst_text_to_sql',
      'input': {'experimental': {'agent_request_id': '875c91f0-4163-4adf-b687-db0627b56aa5'},
       'query': 'For Gamersupps, what is AOV for the last 30 days?'}}},
    {'index': 0,
     'type': 'tool_results',
     'tool_results': {'tool_use_id': 'tooluse_6a8ba8f98bd047d2bdaeed',
      'type': 'cortex_analyst_text_to_sql',
      'content': [{'index': 0,
        'type': 'json',
        'json': {'sql': "WITH __client_analytics_daily_report AS (\n  SELECT\n    brand,\n    report_date,\n    orders_placed,\n    order_total_charged_minus_dt_usd\n  FROM ob_analytics.production.client_analytics_daily_report\n), orders_last_30d AS (\n  SELECT\n    report_date,\n    order_total_charged_minus_dt_usd,\n    orders_placed\n  FROM __client_analytics_da

In [112]:
def process_bot_message(response, summary, debug=False):
    """
    Process the response json returned by Cortex Agent API
    
    """
    
    if not response:
        return
        
    bot_text_message = ""  # Accumulate partial text
    search_results = []
    sql_queries = []
    suggestions = []
    df_sql = pd.DataFrame([])
    final_answer = []
    error_info = None

    # for debug
    if debug:
        yield response
    
    # Display the entire assistant reply as a chat message
    for item in response:
        if item["event"] == "error":
            # If an error occurred mid-stream
            error_info = item["data"]
            break
        elif item['event'] == 'done':
            break
        if 'data' in item and 'delta' in item['data']:
            content = item['data']['delta']['content']
            for obj in content:
                if obj['type'] == 'tool_use':
                    tool_name = obj.get("tool_use", {}).get("name", "Unknown Tool")
                    st.markdown(f":mag_right: :blue[**AI Agent**] is using the tool: `{tool_name}`")
                elif obj['type'] == 'tool_results':
                    result_json = obj.get("tool_results", {}).get("content", [{}])[0].get("json", {})

                    # Grab interesting parts
                    _search = result_json.get("searchResults", [])
                    _sql = result_json.get("sql", [])
                    _suggestions = result_json.get("suggestions", [])
                    _assistant_text = result_json.get("text", "")

                    if _assistant_text:
                        st.markdown(_assistant_text)
                    search_results.extend(_search)
                    sql_queries.extend(_sql)
                    suggestions.extend(_suggestions)

                    if _sql:
                        df_sql = run_snowflake_query(_sql)
                        if not df_sql.empty:
                            prompt_sql_to_english = create_prompt_summarize_cortex_analyst_results(summary, df_sql, _sql[:-1])  
                            final_answer = execute_cortex_complete_api(prompt_sql_to_english)
                            st.write(final_answer)  
                elif obj['type'] == "text":
                    # Plain text from the assistant
                    bot_text_message += obj["text"]
        if bot_text_message:
            st.write(bot_text_message)
            # ---- handle citation ----
            citation_pattern = r'【†(\d+)†】'
            cited_ids_in_text = re.findall(citation_pattern, bot_text_message)
            cited_ids_in_text = set(int(x) for x in cited_ids_in_text)
            cited_docs = [
                doc for doc in search_results
                if doc.get('source_id') in cited_ids_in_text
            ]
            if cited_docs:
                st.markdown("---")
                st.markdown("##### Citation References")
                for doc in cited_docs:
                    doc_id = doc["source_id"]
                    doc_text = doc["text"]
                    with st.expander(f"Citation 【†{doc_id}†】"):
                        st.write(doc_text)
         # --- handle error ----               
        if error_info:
            error_code = error_info.get('code', 'Unknown Code')
            error_msg = error_info.get('message', 'Unknown Error')
            st.error(f"**Error Code:** {error_code}\n\n**Message:** {error_msg}")
                    
    st.session_state.messages.append({
        "role": "assistant",
        "text": bot_text_message,
        #"searchResults": search_results,
        #"sql": sql_queries,
        "df_sql": df_sql,
        "answer_english": final_answer,
        "suggestions": suggestions,
        "type": "error" if error_info else "assistant"
    })

    if st.session_state.debug:
        st.write(st.session_state.messages)

In [113]:
import json
from typing import Any, AsyncIterator, Dict, Optional, List, Tuple
import asyncio
import aiohttp
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse


In [114]:
def snowflake_agent_call(jwt_token:str,snowflake_account:str,prompt:str):
    payload = {
    "model": "claude-3-5-sonnet",
    "response_instruction": prompt,
    "messages": [          
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "For Gamersupps, what is AOV for the last 30 days?"
                }
            ]
        }
    ],
    "tools": [
        {
            "tool_spec": {
                "type": "cortex_analyst_text_to_sql",
                "name": "analyze_data"
            }
        }
        
    ],
    "tool_resources": {
        "analyze_data": {"semantic_model_file": "@OB_AI.CHATBOT.APP/client_analytics.yaml"}
    }
    } 
    url = f"https://{snowflake_account}.snowflakecomputing.com/api/v2/cortex/agent:run"
    headers = {
        "Authorization": f"Bearer {jwt_token}",
        "X-Snowflake-Authorization-Token-Type": "KEYPAIR_JWT",
        "Content-Type": "application/json"
    }

In [115]:
async def astream_sse(jwt_token:str,snowflake_account:str,prompt:str):
    url = f"https://{snowflake_account}.snowflakecomputing.com/api/v2/cortex/agent:run"
    payload = {
    "model": "claude-3-5-sonnet",
    "response_instruction": prompt,
    "messages": [          
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "For Gamersupps, what is AOV for the last 30 days?"
                }
            ]
        }
    ],
    "tools": [
        {
            "tool_spec": {
                "type": "cortex_analyst_text_to_sql",
                "name": "analyze_data"
            }
        }
        
    ],
    "tool_resources": {
        "analyze_data": {"semantic_model_file": "@OB_AI.CHATBOT.APP/client_analytics.yaml"}
    }
    } 
    headers = {
        "Authorization": f"Bearer {jwt_token}",
        "X-Snowflake-Authorization-Token-Type": "KEYPAIR_JWT",
        "Content-Type": "application/json"
    }
    async with aiohttp.ClientSession() as session:
        async with session.post(url, headers=headers, json=payload, timeout=aiohttp.ClientTimeout(total=None)) as resp:
            event, data_lines, event_id = "message", [], None
            async for raw_bytes in resp.content:
                for ch in raw_bytes.splitlines():
                    line = ch.decode("utf-8")
                    if line.startswith(":"):
                        continue
                    if not line:
                        if data_lines:
                            yield {"event": event, "data": "\n".join(data_lines), "id": event_id}
                        event, data_lines = "message", []
                        continue
                    if ":" in line:
                        field, value = line.split(":", 1)
                        value = value[1:] if value.startswith(" ") else value
                    else:
                        field, value = line, ""
                    if field == "event":
                        event = value
                    elif field == "data":
                        data_lines.append(value)
                    elif field == "id":
                        event_id = value
            if data_lines:
                yield {"event": event, "data": "\n".join(data_lines), "id": event_id}


In [116]:
async def main():
    async for event in astream_sse(jwt_token,snowflake_account,'You have to always be friendly and helpful'):
        print(event.get("event"))

In [117]:
await main()

Exception ignored in: <coroutine object main at 0x000001BDFA899B40>
Traceback (most recent call last):
  File "<string>", line 1, in <lambda>
KeyError: '__import__'
Exception ignored in: <coroutine object main at 0x000001BDFA899B40>
Traceback (most recent call last):
  File "<string>", line 1, in <lambda>
KeyError: '__import__'


CancelledError: 

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal, Optional
from datetime import datetime

class chat(BaseModel):
    role: Literal['system', 'user', 'assistant',''] = Field(..., description="The role of the message")
    message: str = Field(..., description="The content of the message")


class prompt(BaseModel):
    curret_prompt: str
    brand: str
    history: Optional[list[chat]] = None

class output(BaseModel):
    type: Literal['intermediate','final'] = Field(..., description="The type of the message")
    role: Literal['assistant'] = Field(..., description="The role of the message")
    message: str = Field(..., description="The content of the message")
    timestamp: datetime = Field(..., description="The timestamp of the message")





In [1]:





import json
json.loads("""{"id":"msg_001","object":"message.delta","delta":{"content":[{"index":0,"type":"tool_use","tool_use":{"tool_use_id":"tooluse_4c3f55a950c94ae1829a07","name":"analyze_data","type":"cortex_analyst_text_to_sql","input":{"experimental":{"agent_request_id":"cbc8d365-1812-4739-a164-df04d2a5c56d"},"query":"For Gamersupps, what is AOV for the last 30 days?"}}},{"index":0,"type":"tool_results","tool_results":{"tool_use_id":"tooluse_4c3f55a950c94ae1829a07","type":"cortex_analyst_text_to_sql","content":[{"index":0,"type":"json","json":{"sql":"WITH __client_analytics_orders AS (\\n  SELECT\\n    source_order_name,\\n    brand,\\n    order_placement_date,\\n    total_charged_minus_dt_usd\\n  FROM ob_analytics.production.client_analytics_orders\\n), orders_last_30d AS (\\n  SELECT\\n    ROUND(SUM(total_charged_minus_dt_usd), 2) AS total_net_sales,\\n    COUNT(DISTINCT source_order_name) AS total_orders\\n  FROM __client_analytics_orders\\n  WHERE\\n    brand = \'gamersupps\' AND order_placement_date >= DATEADD(DAY, -30, CURRENT_DATE)\\n)\\nSELECT\\n  total_net_sales,\\n  total_orders,\\n  ROUND(total_net_sales / NULLIF(NULLIF(total_orders, 0), 0), 2) AS aov\\nFROM orders_last_30d\\n -- Generated by Cortex Analyst\\n;","text":"This is our interpretation of your question:\\n\\nFor Gamersupps brand, what is the Average Order Value (AOV) based on net sales (total_charged_minus_dt_usd) for orders placed in the past 30 days from current date (2025-08-19)?","verified_query_used":false}}],"status":"success","name":"analyze_data"}}]}}""")




{'id': 'msg_001',
 'object': 'message.delta',
 'delta': {'content': [{'index': 0,
    'type': 'tool_use',
    'tool_use': {'tool_use_id': 'tooluse_4c3f55a950c94ae1829a07',
     'name': 'analyze_data',
     'type': 'cortex_analyst_text_to_sql',
     'input': {'experimental': {'agent_request_id': 'cbc8d365-1812-4739-a164-df04d2a5c56d'},
      'query': 'For Gamersupps, what is AOV for the last 30 days?'}}},
   {'index': 0,
    'type': 'tool_results',
    'tool_results': {'tool_use_id': 'tooluse_4c3f55a950c94ae1829a07',
     'type': 'cortex_analyst_text_to_sql',
     'content': [{'index': 0,
       'type': 'json',
       'json': {'sql': "WITH __client_analytics_orders AS (\n  SELECT\n    source_order_name,\n    brand,\n    order_placement_date,\n    total_charged_minus_dt_usd\n  FROM ob_analytics.production.client_analytics_orders\n), orders_last_30d AS (\n  SELECT\n    ROUND(SUM(total_charged_minus_dt_usd), 2) AS total_net_sales,\n    COUNT(DISTINCT source_order_name) AS total_orders\n  F

In [19]:
from pydantic import BaseModel, Field
from typing import Literal, Optional

In [20]:
class chat(BaseModel):
    role: Literal['system', 'user', 'assistant',''] = Field(..., description="The role of the message")
    message: str = Field(..., description="The content of the message, only user and assistant text messages are needed")

class ChatRequest(BaseModel):
    prompt: str = Field(..., description="The prompt to send to the chatbot")
    brand: str = Field(..., description="The brandcode to guardrail data")
    history: Optional[list[chat]] = Field(None, description="The history of the chat")

In [21]:
chat1=chat(role='user',message='For Gamersupps, what is AOV for the last 30 days?')
chat2=chat(role='assistant',message='For Gamersupps, the AOV for the last 30 days is 100.')

In [22]:
request = ChatRequest(prompt='For Gamersupps, what is AOV for the last 30 days?', brand='gamersupps', history=[chat1,chat2])

In [28]:
[chat.model_dump() for chat in request.history]

[{'role': 'user',
  'message': 'For Gamersupps, what is AOV for the last 30 days?'},
 {'role': 'assistant',
  'message': 'For Gamersupps, the AOV for the last 30 days is 100.'}]

In [35]:
chat_history=[{"role":"user","message":"which region we spent the most in June 2025?"},{"role":"assistant","message":"Based on the data:Marketing Spend Analysis for Drmtlgy (June 2025):Canada was the region with the highest marketing spend at $96,511.57"}]

In [36]:
import json
json.loads(chat_history)

TypeError: the JSON object must be str, bytes or bytearray, not list

In [31]:
start_index = max(0, len(chat_history) -6) # look back 6 messages at most
context_history = chat_history[start_index:]

In [32]:
context_history

[{'role': 'user', 'message': 'which region we spent the most in June 2025?'},
 {'role': 'assistant',
  'message': 'Based on the data:Marketing Spend Analysis for Drmtlgy (June 2025):Canada was the region with the highest marketing spend at $96,511.57'}]